In [5]:
import pandas as pd
import numpy as np
import csv
import os
import warnings
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from pyproj import Transformer

warnings.filterwarnings('ignore')

# 1. IDENTIFICACIÓN DEL ENTORNO Y CARGA
ruta = 'accidentes_con_trafico_final.csv'

if not os.path.exists(ruta):
    try:
        from google.colab import files
        print("Entorno Colab detectado. Por favor, sube el archivo:")
        uploaded = files.upload()
    except ImportError:
        print(f"❌ Error: No se encuentra el archivo '{ruta}' en tu carpeta local.")
        print("Asegúrate de que el CSV esté en la misma carpeta que este cuaderno.")

# 2. LECTOR ROBUSTO
def lector_robusto(path):
    datos = []
    with open(path, 'r', encoding='latin1') as f:
        reader = csv.reader(f)
        next(reader) 
        for row in reader:
            if len(row) == 18:
                datos.append(row)
            elif len(row) > 18:
                inicio, final = row[:12], row[-5:]
                direccion_arreglada = " ".join(row[12:-5])
                datos.append(inicio + [direccion_arreglada] + final)
    return pd.DataFrame(datos, columns=columnas_header)

columnas_header = ["fecha","hora","dia_semana","distrito","num_expediente","tipo_accidente",
                   "tipo_vehiculo","sexo","rango_edad","estado_meteorologico",
                   "coordenada_x_utm","coordenada_y_utm","direccion_unica","es_festivo",
                   "id_sensor_cercano","intensidad","ocupacion","vmed"]

print("⏳ Procesando datos...")
df = lector_robusto(ruta)

# 3. LIMPIEZA E IMPUTACIÓN
df['coordenada_x_utm'] = pd.to_numeric(df['coordenada_x_utm'], errors='coerce')
df['coordenada_y_utm'] = pd.to_numeric(df['coordenada_y_utm'], errors='coerce')
df['es_festivo'] = pd.to_numeric(df['es_festivo'], errors='coerce').fillna(0)
df['fecha'] = pd.to_datetime(df['fecha'], format='%Y-%m-%d', errors='coerce')
df = df.dropna(subset=['fecha'])
df['mes_dia'] = df['fecha'].dt.strftime('%m-%d')
df['dia_semana'] = df['dia_semana'].str.lower().str.strip()

coords_calle = df.groupby('direccion_unica')[['coordenada_x_utm', 'coordenada_y_utm']].transform('mean')
df['coordenada_x_utm'] = df['coordenada_x_utm'].fillna(coords_calle['coordenada_x_utm'])
df['coordenada_y_utm'] = df['coordenada_y_utm'].fillna(coords_calle['coordenada_y_utm'])

df_madrid = df.dropna(subset=['coordenada_x_utm', 'coordenada_y_utm']).copy()

# Conversión GPS y Filtro Madrid
transformer = Transformer.from_crs("epsg:25830", "epsg:4326", always_xy=True)
lons, lats = transformer.transform(df_madrid['coordenada_x_utm'].values, df_madrid['coordenada_y_utm'].values)
df_madrid['lat'], df_madrid['lon'] = lats, lons
df_madrid = df_madrid[(df_madrid['lat'] > 40.3) & (df_madrid['lat'] < 40.6) & (df_madrid['lon'] > -3.9) & (df_madrid['lon'] < -3.5)]

print(f"✅ Dataset listo. Filas útiles: {len(df_madrid)}")

⏳ Procesando datos...
✅ Dataset listo. Filas útiles: 152815


In [6]:
# 1. Intervalos Horarios
def asignar_intervalo(hora_str):
    try:
        h = int(str(hora_str).split(':')[0])
        if 0 <= h < 7: return '01_Madrugada'
        elif 7 <= h < 9: return '02_Punta_Entrada'
        elif 9 <= h < 15: return '03_Mañana_Trabajo'
        elif 15 <= h < 17: return '04_Salida_Comida'
        elif 17 <= h < 21: return '05_Tarde_Ocio'
        else: return '06_Noche'
    except: return 'Desconocido'

df_madrid['intervalo_hora'] = df_madrid['hora'].apply(asignar_intervalo)

# 2. Cálculo de Riesgo (Z-Score + Scaling)
def calcular_riesgo(df_input, cols):
    agg = df_input.groupby(cols).size().reset_index(name='n_acc')
    agg['z'] = StandardScaler().fit_transform(agg[['n_acc']])
    agg['riesgo'] = MinMaxScaler(feature_range=(0, 10)).fit_transform(agg[['z']]).round(2)
    return agg

r_calendario = calcular_riesgo(df_madrid, ['direccion_unica', 'mes_dia', 'intervalo_hora'])
r_semanal = calcular_riesgo(df_madrid[df_madrid['es_festivo'] == 0], ['direccion_unica', 'dia_semana', 'intervalo_hora'])

# 3. UNIFICACIÓN EN EL DATASET PRINCIPAL
df_madrid = df_madrid.merge(r_semanal[['direccion_unica', 'dia_semana', 'intervalo_hora', 'riesgo']], 
                            on=['direccion_unica', 'dia_semana', 'intervalo_hora'], how='left').rename(columns={'riesgo': 'r_sem'})

df_madrid = df_madrid.merge(r_calendario[['direccion_unica', 'mes_dia', 'intervalo_hora', 'riesgo']], 
                            on=['direccion_unica', 'mes_dia', 'intervalo_hora'], how='left').rename(columns={'riesgo': 'r_cal'})

# Armonización final
df_madrid['riesgo_total'] = df_madrid[['r_sem', 'r_cal']].mean(axis=1).fillna(df_madrid['r_cal']).fillna(0).round(2)

print("🚀 Variable 'riesgo_total' generada en el dataset principal.")

🚀 Variable 'riesgo_total' generada en el dataset principal.


In [9]:
import ipywidgets as widgets
from ipywidgets import interact
import folium
import branca.colormap as cm

# 1. Definición de la Paleta de Colores de Riesgo (0-10)
colormap = cm.LinearColormap(colors=['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c'], vmin=0, vmax=10)
colormap.caption = 'Índice de Riesgo Unificado (0-10)'

# 2. Función Principal del Visualizador
@interact(modo=['Por Día de la Semana', 'Por Fecha Específica'])
def control_principal(modo):
    # Definimos las opciones dinámicas según el modo seleccionado
    if modo == 'Por Día de la Semana':
        opciones = ['lunes', 'martes', 'miercoles', 'jueves', 'viernes', 'sabado', 'domingo']
        desc_tiempo = 'Día:'
    else:
        opciones = sorted(df_madrid['mes_dia'].unique().tolist())
        desc_tiempo = 'Fecha:'

    # Segundo nivel de interactividad: Selección de Tiempo y Franja Horaria
    @interact(seleccion=widgets.Dropdown(options=opciones, description=desc_tiempo), 
              hora=widgets.Dropdown(options=sorted(df_madrid['intervalo_hora'].unique()), description='Franja Horaria:'))
    def dibujar_mapa(seleccion, hora):
        # APLICACIÓN DE FILTROS
        if modo == 'Por Día de la Semana':
            filtro = (df_madrid['dia_semana'] == seleccion)
        else:
            filtro = (df_madrid['mes_dia'] == seleccion)
        
        # Agrupamos por calle para representar tramos, no puntos individuales
        # Calculamos la media de riesgo_total y accidentes para el Tooltip
        data_plot = df_madrid[filtro & (df_madrid['intervalo_hora'] == hora)].groupby(['direccion_unica', 'lat', 'lon']).agg(
            riesgo_medio=('riesgo_total', 'mean'),
            total_accidentes=('num_expediente', 'count')
        ).reset_index()
        
        if data_plot.empty:
            print("Sin registros para esta combinación de filtros.")
            return

        # CREACIÓN DEL MAPA BASE (Centrado en Madrid)
        m = folium.Map(location=[40.4167, -3.7033], zoom_start=13, tiles='cartodbpositron')
        colormap.add_to(m) # Añadimos la leyenda de colores
        
        # AÑADIR PUNTOS DE RIESGO
        for _, row in data_plot.iterrows():
            # El radio del círculo aumenta ligeramente si el riesgo es alto
            radio_punto = 6 + (row['riesgo_medio'] / 2)
            
            folium.CircleMarker(
                location=[row['lat'], row['lon']],
                radius=radio_punto,
                color=colormap(row['riesgo_medio']),
                fill=True,
                fill_opacity=0.8,
                # ESTE ES EL OVERLAY/TOOLTIP (Información al pasar el ratón)
                tooltip=(f"<b>CALLE:</b> {row['direccion_unica']}<br>"
                         f"<b>RIESGO UNIFICADO:</b> {row['riesgo_medio']:.2f}/10<br>"
                         f"<b>ACCIDENTES REGISTRADOS:</b> {row['total_accidentes']}")
            ).add_to(m)
        
        display(m)

interactive(children=(Dropdown(description='modo', options=('Por Día de la Semana', 'Por Fecha Específica'), v…

In [11]:
# Seleccionamos una muestra significativa: algunos festivos y algunos días normales
muestra = pd.concat([
    df_madrid[df_madrid['es_festivo'] == 1].head(),
    df_madrid[df_madrid['es_festivo'] == 0].head()
])

# Mostramos solo las columnas de interés
columnas_interes = ['fecha', 'dia_semana', 'intervalo_hora', 'direccion_unica', 'es_festivo', 'riesgo_total']

print("📊 MUESTRA DEL DATASET ENRIQUECIDO (VISTA MINABLE):")
display(muestra[columnas_interes].sort_values(by='fecha'))

📊 MUESTRA DEL DATASET ENRIQUECIDO (VISTA MINABLE):


,fecha,dia_semana,intervalo_hora,direccion_unica,es_festivo,riesgo_total
0,2016-01-01,viernes,01_Madrugada,AVENIDA DEL MARQUES DE CORBERA NUM 7,1,0.37
1,2016-01-01,viernes,01_Madrugada,AVENIDA DEL MARQUES DE CORBERA NUM 7,1,0.37
2,2016-01-01,viernes,01_Madrugada,CALLE DE OCAÃA - CALLE DE VALMOJADO 0,1,0.00
3,2016-01-01,viernes,01_Madrugada,CALLE DEL DOCTOR ESQUERDO NUM 219,1,0.00
4,2016-01-01,viernes,03_Mañana_Trabajo,PASEO DE LOS TALLERES NUM 70,1,0.00
16,2016-01-02,sabado,01_Madrugada,AUTOVIA M-30 CALZADA 2 KM. 30400,0,0.00
17,2016-01-02,sabado,03_Mañana_Trabajo,CALLE DE EMBAJADORES NUM 66,0,0.00
18,2016-01-02,sabado,03_Mañana_Trabajo,CALLE DE HERMOSILLA NUM 82,0,0.00
19,2016-01-02,sabado,03_Mañana_Trabajo,CALLE DE ATOCHA - PLAZA DE JACINTO BENAVENTE 0,0,0.00
20,2016-01-02,sabado,03_Mañana_Trabajo,AVENIDA DEL GENERAL FANJUL - CALLE DE GUAREÃA 0,0,0.37


In [12]:
df_madrid.head()

,fecha,hora,dia_semana,distrito,num_expediente,tipo_accidente,tipo_vehiculo,sexo,rango_edad,estado_meteorologico,...,intensidad,ocupacion,vmed,mes_dia,lat,lon,intervalo_hora,r_sem,r_cal,riesgo_total
0,2016-01-01,01:00:00,viernes,ciudad lineal,2016/437,colisiÃ³n doble,turismo,mujer,de 30 a 34 anos,despejado,...,,,;;,01-01,40.423326,-3.655565,01_Madrugada,NaN,0.37,0.37
1,2016-01-01,01:00:00,viernes,ciudad lineal,2016/437,colisiÃ³n doble,motocicleta,hombre,de 25 a 29 aÃ±os,despejado,...,,,;;,01-01,40.423326,-3.655565,01_Madrugada,NaN,0.37,0.37
2,2016-01-01,03:00:00,viernes,latina,2016/6,atropello,turismo,mujer,de 30 a 34 anos,lluvia dÃ©bil,...,,,;;,01-01,40.386291,-3.753779,01_Madrugada,NaN,0.00,0.00
3,2016-01-01,05:00:00,viernes,retiro,2016/12,otras causas,auto-taxi,hombre,de 60 a 64 aÃ±os,despejado,...,,,;;,01-01,40.401495,-3.674295,01_Madrugada,NaN,0.00,0.00
4,2016-01-01,12:00:00,viernes,villaverde,2016/26,atropello,turismo,hombre,de 45 a 49 aÃ±os,despejado,...,175.0,2.75,0.0;;,01-01,40.347352,-3.705812,03_Mañana_Trabajo,NaN,0.00,0.00


In [ ]:
# Definimos el nombre del archivo final
archivo_exportar = 'madrid_minable_view_final.csv'

# Seleccionamos las columnas que realmente necesitan tus compañeros para la IA
# Evitamos columnas redundantes para que el archivo no pese demasiado
columnas_para_ia = [
    'fecha', 'dia_semana', 'intervalo_hora', 'distrito', 
    'tipo_accidente', 'tipo_vehiculo', 'sexo', 'rango_edad', 
    'estado_meteorologico', 'es_festivo', 'lat', 'lon', 'riesgo_total'
]

# Guardamos el dataset
df_madrid[columnas_para_ia].to_csv(archivo_exportar, index=False)

print(f"✅ Archivo '{archivo_exportar}' generado con éxito.")

✅ Archivo 'madrid_minable_view_final.csv' generado con éxito.
📥 Ahora puedes descargarlo desde el icono de la carpeta (izquierda) o usar el siguiente comando:
